# 01B — ETL Nguồn OULAD → DataMart Chuẩn Hóa

**Dataset:** Open University Learning Analytics Dataset (OULAD)  
**Input:** `data/processed/student_features_labeled.csv` (đầu ra của `01_data_cleaning.ipynb`)  
**Output:** `data/processed/oulad_datamart.csv`  
**Vai trò OOD:** Tập **TRAIN / VAL** — Dữ liệu này được dùng để học trọng số mô hình.

---

### Sơ đồ cột nguồn → DataMart

| Cột DataMart | Cột nguồn | Ghi chú |
|---|---|---|
| `student_code` | `id_student` | Prefix `OU_` |
| `academic_year_order` | `num_of_prev_attempts` | +1 |
| `admission_score_norm` | `education_num` | `/4.0` — proxy trình độ học vấn |
| `priority_flag` | `disability_num` | 0/1 |
| `credits_registered` | `studied_credits` | Trực tiếp |
| `credit_completion_ratio` | `final_result` | 1.0 nếu Pass/Distinction |
| `gpa_current_semester` | `avg_score` | Piecewise (qua môn=40→GPA 2.0) |
| `cpa_cumulative_score` | `avg_score` | Proxy = GPA (không có đa kỳ) |
| `lms_total_clicks` | `total_clicks` | Từ VLE — đã xử lý ở 01 |
| `lms_active_days` | `active_days` | Từ VLE |
| `lms_late_submits` | `n_late` | Số bài nộp trễ |
| `lms_avg_quiz` | `avg_tma_score / 10` | TMA scale 100 → 10 |
| `target_risk_level` | `risk_level` | Đã gán nhãn ở `01_data_cleaning` |


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

# Tự động xác định thư mục gốc
RAW_DIR       = 'data/raw/'       if os.path.basename(os.getcwd()) != 'notebooks' else '../data/raw/'
PROCESSED_DIR = 'data/processed/' if os.path.basename(os.getcwd()) != 'notebooks' else '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Setup hoàn tất")
print(f"  RAW_DIR       : {os.path.abspath(RAW_DIR)}")
print(f"  PROCESSED_DIR : {os.path.abspath(PROCESSED_DIR)}")


## 1. Load Dữ Liệu Đầu Vào

In [ ]:
# Đọc file đã qua 01_data_cleaning.ipynb
df = pd.read_csv(PROCESSED_DIR + 'student_features_labeled.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
print("=== 3 dòng đầu ===")
display(df.head(3))
print()
print("=== Phân phối final_result ===")
print(df['final_result'].value_counts())
print()
print("=== Phân phối risk_label (nhãn đã gán) ===")
print(df['risk_label'].value_counts())
print()
print("=== Thống kê avg_score (0-100) ===")
print(df['avg_score'].describe())
print()
print("=== Kiểm tra NaN ===")
nan_cols = df.isnull().sum()
print(nan_cols[nan_cols > 0] if nan_cols.any() else "Không có NaN")


## 2. Hàm Piecewise GPA Alignment

> **Lý do:** OULAD dùng thang 0–100, ngưỡng qua môn = 40. Ánh xạ tuyến tính đơn giản cho ngưỡng qua môn ở GPA=1.6, trong khi UCI có ngưỡng ở GPA=2.0. Piecewise đảm bảo ngưỡng **luôn = 2.0** ở cả hai hệ thống.

In [ ]:
def map_oulad_gpa(s):
    """Piecewise: OULAD 0-100 → GPA 0-4, qua môn (40) → 2.0"""
    if pd.isna(s): return 0.0
    if s < 40:
        return (s / 40.0) * 2.0
    return 2.0 + ((s - 40.0) / 60.0) * 2.0

# Kiểm tra tại các mốc chuẩn
test_scores = [0, 20, 40, 55, 70, 85, 100]
print("Score → GPA (Piecewise):")
for s in test_scores:
    marker = " ← Ngưỡng qua môn" if s == 40 else ""
    print(f"  score={s:3d} → GPA={map_oulad_gpa(s):.3f}{marker}")


## 3. Ánh Xạ → 22 Cột DataMart

In [ ]:
out = pd.DataFrame()

# ── IDENTITY ──────────────────────────────────────────────────────────────────
out['student_code'] = 'OU_' + df['id_student'].astype(str)
out['academic_year_order'] = df['num_of_prev_attempts'] + 1

# ── ADMISSION — Proxy từ trình độ học vấn (education_num: 0-4 từ 01_data_cleaning) ──
# education_num: 0=No Formal, 1=Lower A, 2=A Level, 3=HE Qual, 4=Post Grad
out['admission_score_norm'] = (df['education_num'] / 4.0).clip(0, 1)

# ── STUDENT PROFILE ────────────────────────────────────────────────────────────
out['priority_flag'] = df['disability_num']  # 0/1 đã encode ở 01_data_cleaning

# ── CREDITS ───────────────────────────────────────────────────────────────────
out['credits_registered']     = df['studied_credits']
out['credit_completion_ratio'] = df['final_result'].apply(
    lambda x: 1.0 if x in ['Pass', 'Distinction'] else 0.0)
# OULAD không track nợ tín chỉ từng môn riêng biệt
out['cumulative_credit_debt']  = df['final_result'].apply(
    lambda x: 1 if x == 'Fail' else 0)

# ── GPA (Piecewise Aligned) ───────────────────────────────────────────────────
out['gpa_current_semester'] = df['avg_score'].apply(map_oulad_gpa)
# OULAD không có lịch sử đa kỳ → dùng GPA hiện tại làm proxy
out['cpa_cumulative_score'] = out['gpa_current_semester']
# Không có thông tin tăng trưởng liên kỳ
out['cpa_growth_delta'] = 0.0

# ── ACADEMIC PERFORMANCE ──────────────────────────────────────────────────────
out['failed_courses_count'] = df['final_result'].apply(lambda x: 1 if x == 'Fail' else 0)

# training_point_norm: OULAD không có điểm rèn luyện → NaN
# (sẽ được xử lý bởi SimpleImputer trong pipeline Modeling)
out['training_point_norm'] = np.nan

# Không có thông tin học phí
out['tuition_debt_flag'] = 0

# ── LMS — Dữ liệu tương tác thực từ VLE (OULAD có đầy đủ) ────────────────────
out['lms_total_clicks'] = df['total_clicks']
out['lms_active_days']  = df['active_days']
out['lms_late_submits'] = df['n_late']
# avg_tma_score thang 100 → [0, 10] để đồng nhất với quiz score
out['lms_avg_quiz']     = (df['avg_tma_score'] / 10.0).clip(0, 10)

# ── WARNING HISTORY ────────────────────────────────────────────────────────────
out['previous_academic_warnings'] = df['num_of_prev_attempts']

# ── MODEL OUTPUT (placeholder) ────────────────────────────────────────────────
out['risk_score_continuous'] = 0.0

# ── TARGET — Dùng lại nhãn đã gán ở 01_data_cleaning ─────────────────────────
out['target_risk_level'] = df['risk_level']
out['target_risk_name']  = df['risk_label']
out['dataset_source']    = 'OULAD'

# Loại bỏ trùng lặp: giữ 1 dòng mỗi sinh viên
out = out.drop_duplicates(subset=['student_code']).reset_index(drop=True)

print(f"Shape sau ETL: {out.shape}")
print(f"Cột: {list(out.columns)}")


## 4. Kiểm Tra Chất Lượng

In [ ]:
print("=== Phân phối nhãn rủi ro ===")
print(out['target_risk_name'].value_counts())
print()

print("=== Thống kê GPA sau Piecewise ===")
gpa_stats = out['gpa_current_semester'].describe()
print(gpa_stats)
pct_pass = (out['gpa_current_semester'] >= 2.0).mean() * 100
print(f"Tỷ lệ GPA ≥ 2.0 (qua môn): {pct_pass:.1f}%")
print()

print("=== Thống kê admission_score_norm ===")
print(out['admission_score_norm'].value_counts().sort_index())
print(f"Phương sai: {out['admission_score_norm'].var():.4f}  ← phải > 0")
print()

print("=== NaN count theo cột ===")
nan_check = out.isnull().sum()
nan_cols = nan_check[nan_check > 0]
if len(nan_cols) > 0:
    print(nan_cols)
    print("\n⚠️ training_point_norm = NaN là có chủ đích (sẽ impute ở Modeling)")
else:
    print("Không có NaN nào ngoài dự kiến")
print()

print("=== LMS distribution ===")
print(out[['lms_total_clicks','lms_active_days','lms_late_submits','lms_avg_quiz']].describe())


## 5. Lưu File

In [ ]:
out_path = PROCESSED_DIR + 'oulad_datamart.csv'
out.to_csv(out_path, index=False)
print(f"✅ Đã lưu: {out_path}")
print(f"   Shape : {out.shape}")
print(f"   Size  : {os.path.getsize(out_path)/1024:.1f} KB")
